In [ ]:
# Install required packages
!pip install -q unsloth transformers datasets trl accelerate

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
import pickle
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

## 1. Load Data

In [ ]:
# Load training data
train_df = pd.read_csv('data/train.csv')
print(f"Training data shape: {train_df.shape}")
print(f"\nColumns: {train_df.columns.tolist()}")
print(f"\nRoot Cause distribution:")
print(train_df['Root_Cause'].value_counts())

## 2. Train ML Model (or Load Existing)

First, we train a baseline ML model that will provide predictions to enhance the LLM prompts.

In [ ]:
# Prepare features for ML model
def prepare_ml_features(df):
    """
    Prepare features for traditional ML model.
    Encodes categorical variables.
    """
    df_encoded = df.copy()
    
    # Encode categorical features
    categorical_cols = ['Symptom_1', 'Symptom_2', 'Symptom_3', 
                        'Region', 'Network_Type', 'Incident_Time']
    
    encoders = {}
    for col in categorical_cols:
        le = LabelEncoder()
        df_encoded[f'{col}_encoded'] = le.fit_transform(df[col].astype(str))
        encoders[col] = le
    
    # Select feature columns
    feature_cols = [f'{col}_encoded' for col in categorical_cols] + ['Duration_Minutes']
    
    return df_encoded[feature_cols], encoders

# Prepare features
X, encoders = prepare_ml_features(train_df)
y = train_df['Root_Cause']

# Encode target
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)

print(f"Feature shape: {X.shape}")
print(f"Number of classes: {len(target_encoder.classes_)}")

In [ ]:
# Train ML model (Random Forest)
from sklearn.metrics import classification_report, accuracy_score

# Split for ML model validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, test_size=0.2, random_state=SEED, stratify=y_encoded
)

# Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    random_state=SEED,
    n_jobs=-1
)

print("Training ML model...")
rf_model.fit(X_train, y_train)

# Evaluate
y_pred = rf_model.predict(X_val)
print(f"\nML Model Validation Accuracy: {accuracy_score(y_val, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred, target_names=target_encoder.classes_))

In [ ]:
# Get ML predictions with confidence scores for entire training set
ml_predictions = rf_model.predict(X)
ml_predictions_proba = rf_model.predict_proba(X)

# Decode predictions to root cause names
ml_predictions_decoded = target_encoder.inverse_transform(ml_predictions)

# Add to dataframe
train_df['ML_Prediction'] = ml_predictions_decoded
train_df['ML_Confidence'] = ml_predictions_proba.max(axis=1)

print(f"ML predictions added to training data")
print(f"\nSample predictions:")
print(train_df[['Root_Cause', 'ML_Prediction', 'ML_Confidence']].head(10))

## 3. Create Enhanced Prompts with ML Predictions

Now we create prompts that include the ML model's prediction as additional context.

In [ ]:
def format_prompt_with_ml_prediction(row):
    """
    Create a prompt that includes ML model prediction as context.
    The LLM learns to use this as a hint while making its own assessment.
    """
    prompt = f"""<|im_start|>system
You are an expert telecom network troubleshooting assistant. Your task is to analyze network incidents and identify the root cause.
You will be provided with symptoms and an ML model's prediction. Use the ML prediction as a helpful reference, but make your own assessment based on all available information.<|im_end|>
<|im_start|>user
Network Incident Details:

Primary Symptoms:
1. {row['Symptom_1']}
2. {row['Symptom_2']}
3. {row['Symptom_3']}

Incident Context:
- Region: {row['Region']}
- Network Type: {row['Network_Type']}
- Incident Time: {row['Incident_Time']}
- Duration: {row['Duration_Minutes']} minutes

ML Model Analysis:
- Predicted Root Cause: {row['ML_Prediction']}
- Confidence Score: {row['ML_Confidence']:.2%}

Based on the symptoms, context, and ML prediction, identify the root cause of this network incident.<|im_end|>
<|im_start|>assistant
{row['Root_Cause']}<|im_end|>"""
    return prompt

# Create prompts for all training data
train_df['text'] = train_df.apply(format_prompt_with_ml_prediction, axis=1)

print("Sample prompt with ML prediction:")
print("="*80)
print(train_df['text'].iloc[0])
print("="*80)

## 4. Prepare Dataset for Fine-tuning

In [ ]:
from datasets import Dataset

# Split into train and validation
train_data, val_data = train_test_split(
    train_df, test_size=0.15, random_state=SEED, stratify=train_df['Root_Cause']
)

# Create datasets
train_ds = Dataset.from_pandas(train_data[['text']].reset_index(drop=True))
val_ds = Dataset.from_pandas(val_data[['text']].reset_index(drop=True))

print(f"Training samples: {len(train_ds)}")
print(f"Validation samples: {len(val_ds)}")

## 5. Load and Configure Qwen2.5-1.5B Model

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 3300
dtype = None  # Auto-detect
load_in_4bit = True

# Load model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print(f"Model loaded: {model.config.model_type}")
print(f"Model size: ~1.5B parameters")

In [ ]:
# Add LoRA adapters - optimized configuration
model = FastLanguageModel.get_peft_model(
    model,
    r=64,                      # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=128,            # 2x rank
    lora_dropout=0,            # Optimized for unsloth
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=True,           # Rank stabilized LoRA
    loftq_config=None,
)

print("LoRA adapters attached successfully")

## 6. Configure Training - T4 Optimized

In [ ]:
from trl import SFTTrainer, SFTConfig

config = SFTConfig(
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    eos_token="<|im_end|>",
    packing=False,                     # Sequences are already long

    # Batch size - T4 optimized
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,     # Effective batch size = 16
    per_device_eval_batch_size=1,

    # Training schedule
    num_train_epochs=3,
    learning_rate=2e-4,                # Higher for LoRA
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # Logging and evaluation
    logging_steps=100,
    logging_strategy="steps",
    eval_strategy="steps",
    eval_steps=100,

    # Saving
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Optimization
    optim="adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,
    
    # Precision - T4 optimized
    fp16=True,                         # T4 optimized for fp16
    bf16=False,
    tf32=False,
    
    # Efficiency
    group_by_length=True,
    dataloader_num_workers=2,
    
    seed=SEED,
    output_dir="./ensemble_model_checkpoints",
    report_to="none",
)

print("Training configuration set")

In [ ]:
# Initialize trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=config,
)

print("Trainer initialized successfully")

## 7. Train the Model

In [ ]:
# Start training
print("Starting training...")
trainer_stats = trainer.train()

print("\nTraining completed!")
print(f"Final training loss: {trainer_stats.training_loss:.4f}")

## 8. Save Models

In [ ]:
# Save fine-tuned LLM
model.save_pretrained("ensemble_llm_finetuned")
tokenizer.save_pretrained("ensemble_llm_finetuned")

# Save ML model and encoders
with open('ensemble_ml_model.pkl', 'wb') as f:
    pickle.dump({
        'model': rf_model,
        'encoders': encoders,
        'target_encoder': target_encoder
    }, f)

print("Models saved successfully!")
print("- LLM: ensemble_llm_finetuned/")
print("- ML: ensemble_ml_model.pkl")

## 9. Inference Function

In [ ]:
def predict_with_ensemble(test_row, ml_model, encoders, target_encoder, llm_model, tokenizer):
    """
    Make predictions using ensemble approach:
    1. Get ML model prediction
    2. Create prompt with ML prediction
    3. Get LLM final prediction
    """
    # Step 1: Prepare features and get ML prediction
    features = []
    categorical_cols = ['Symptom_1', 'Symptom_2', 'Symptom_3', 
                        'Region', 'Network_Type', 'Incident_Time']
    
    for col in categorical_cols:
        encoded_val = encoders[col].transform([str(test_row[col])])[0]
        features.append(encoded_val)
    features.append(test_row['Duration_Minutes'])
    
    features = np.array(features).reshape(1, -1)
    
    # Get ML prediction
    ml_pred_encoded = ml_model.predict(features)[0]
    ml_pred_proba = ml_model.predict_proba(features)[0]
    ml_prediction = target_encoder.inverse_transform([ml_pred_encoded])[0]
    ml_confidence = ml_pred_proba.max()
    
    # Step 2: Create prompt with ML prediction
    test_row_with_ml = test_row.copy()
    test_row_with_ml['ML_Prediction'] = ml_prediction
    test_row_with_ml['ML_Confidence'] = ml_confidence
    
    prompt = format_prompt_with_ml_prediction(test_row_with_ml)
    
    # Step 3: Get LLM prediction
    FastLanguageModel.for_inference(llm_model)
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.1,
        top_p=0.9,
        do_sample=True,
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract assistant response
    if "<|im_start|>assistant" in response:
        llm_prediction = response.split("<|im_start|>assistant")[-1].strip()
    else:
        llm_prediction = response.split("<|im_end|>")[-1].strip()
    
    return {
        'ml_prediction': ml_prediction,
        'ml_confidence': ml_confidence,
        'llm_prediction': llm_prediction,
        'prompt': prompt
    }

print("Inference function ready")

## 10. Test on Sample Data

In [ ]:
# Test on a validation sample
test_idx = 0
test_sample = val_data.iloc[test_idx]

result = predict_with_ensemble(
    test_sample, 
    rf_model, 
    encoders, 
    target_encoder, 
    model, 
    tokenizer
)

print("Ensemble Prediction Results:")
print("="*80)
print(f"Ground Truth: {test_sample['Root_Cause']}")
print(f"\nML Prediction: {result['ml_prediction']}")
print(f"ML Confidence: {result['ml_confidence']:.2%}")
print(f"\nLLM Final Prediction: {result['llm_prediction']}")
print("="*80)

## 11. Generate Predictions for Test Set

In [ ]:
# Load test data
test_df = pd.read_csv('data/phase_1_test.csv')
print(f"Test data shape: {test_df.shape}")

# Generate predictions
predictions = []

for idx, row in test_df.iterrows():
    if idx % 10 == 0:
        print(f"Processing {idx}/{len(test_df)}...")
    
    result = predict_with_ensemble(row, rf_model, encoders, target_encoder, model, tokenizer)
    predictions.append(result['llm_prediction'])

# Create submission
submission_df = pd.DataFrame({
    'Incident_ID': test_df['Incident_ID'],
    'Root_Cause': predictions
})

submission_df.to_csv('ensemble_submission.csv', index=False)
print("\nSubmission file created: ensemble_submission.csv")
print(submission_df.head())

## Summary

This ensemble approach combines:
- **ML Model**: Fast, structured feature analysis
- **LLM**: Contextual understanding and ability to override ML when needed

Benefits:
1. LLM can leverage ML predictions as hints
2. LLM can correct ML errors based on context
3. Combines strengths of both approaches
4. Often outperforms either model alone